# Semantic Federated — Reprodução dos resultados legados (paper submetido)

Este notebook reproduz a **mesma configuração de experimento** usada para gerar os resultados do artigo submetido/aceito na SBrT 2026 — mesmos datasets, dimensões latentes, níveis de ruído, 5 clientes, 3 rounds e **seed única (42)**, sem particionamento não-IID e sem canais com desvanecimento (só AWGN).

A diferença para o notebook principal (`run_experiments_colab.ipynb`) é só o escopo do experimento (uma seed, a config original do paper) — o pipeline usado é o **atual**, com GPU e salvamento incremental. Isto é, você recupera os números do paper original sem herdar os problemas do código antigo (CPU forçada, sem salvamento incremental, horas de execução).

Antes de rodar: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4)**.

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/GianSE/semantic-federeted.git"
REPO_DIR = "/content/semantic-federeted"

import os

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repositorio ja clonado -- puxando ultimas mudancas (git pull)...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## Persistência via Google Drive

Pasta própria (`semantic-federeted-legacy`) para não misturar com os dados/resultados dos experimentos multi-seed/non-IID/canal do notebook principal.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/semantic-federeted-legacy"
os.makedirs(f"{DRIVE_BASE}/data", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/results", exist_ok=True)

import shutil

for local_name, drive_path in [("data", f"{DRIVE_BASE}/data"), ("results", f"{DRIVE_BASE}/results")]:
    local_path = os.path.join(REPO_DIR, local_name)
    if os.path.islink(local_path):
        continue
    if os.path.isdir(local_path):
        shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f"{local_name}/ -> {drive_path}")

## Rodar a configuração original do paper

Mesmos hiperparâmetros documentados no artigo, só com `--seeds 42` (seed única, como era originalmente) em vez de multi-seed. `--partition iid` e `--channel-type awgn` são os defaults do pipeline atual, mas deixados explícitos aqui para registrar que é exatamente a condição do paper.

In [ ]:
!python main.py \
  --datasets mnist cifar10 \
  --latent-dims 16 32 64 128 \
  --noise-levels 0.0 0.01 0.05 0.1 \
  --num-clients 5 \
  --rounds 3 \
  --seeds 42 \
  --partition iid \
  --channel-type awgn \
  --num-workers 2

## Conferir resultados

In [ ]:
import pandas as pd

df = pd.read_csv("results/data/experiment_results.csv")
df

Compare com os números relatados no paper (ex.: CIFAR-10, L=64 -> acurácia ~0.6509, CR=48x). Os gráficos ficam em `results/plots/` e as tabelas em `results/tables/` -- como é seed única, as barras de erro nos gráficos ficam zeradas (desvio padrão de 1 amostra não existe), o que é esperado e consistente com o que o paper original reportava.